# 에러 해결 기록: `ModuleNotFoundError: No module named 'src'`

- 발생 파일: `notebooks/08_00_작은_데이터_분석_프로젝트_8장.ipynb`
- 발생 코드: `src/preprocessing.py`에 있는 함수들을 노트북에서 import
- 목적: 같은 유형의 에러가 재발했을 때, 무엇을 시도했고 무엇이 실제 원인이었는지 순서대로 참고하기 위한 기록

## 1. 문제 상황

노트북에서 프로젝트 내부 모듈(`src/preprocessing.py`)을 import하려고 했더니 실패했다.

In [ ]:
from src.preprocessing import (
    compare_shapes,
    preprocess_sales_data,
    validate_relationships,
)

**발생한 에러**

```
ModuleNotFoundError: No module named 'src'
```

**원인 요약**: 노트북 파일은 `00_llm-data-analysis-course/notebooks/`에 있고, `src` 패키지는 그 상위 폴더인 `00_llm-data-analysis-course/`에 있다. Jupyter는 기본적으로 **노트북 파일이 있는 폴더를 커널의 현재 작업 디렉터리(cwd)** 로 잡는다. 파이썬은 cwd를 `sys.path`에 자동으로 추가하지만, `src`는 cwd의 하위 폴더가 아니라 형제(sibling) 폴더이기 때문에 import 경로에 잡히지 않는다.

## 2. 시도 1 — 저장소 루트에 `pyproject.toml`을 두고 패키지로 설치

`src`를 매번 `sys.path`에 수동으로 추가하는 대신, 프로젝트를 pip이 인식하는 패키지로 만들어서 어디서 실행하든 `import src`가 되게 하려고 시도했다.

In [ ]:
# 저장소 루트(C:\dev\ai-data-analysis)의 pyproject.toml

[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"

[project]
name = "ai-data-analysis"
version = "0.1.0"
description = "LLM Data Analysis Course"
requires-python = ">=3.11"

[tool.setuptools.packages.find]
where = ["."]
include = ["course_utils*", "src*", "src."]

In [ ]:
# 저장소 루트에서 실행
pip install -e .

**결과**: 설치는 "성공"했지만 노트북에서 여전히 `ModuleNotFoundError`가 그대로 발생했다.

**원인 해설**

`[tool.setuptools.packages.find]`의 `where = ["."]`는 **`pyproject.toml`이 있는 위치(=저장소 루트) 바로 아래**에서 `src`, `course_utils` 폴더를 찾으라는 뜻이다. 그런데 실제 `src` 폴더는 저장소 루트가 아니라 `00_llm-data-analysis-course/src`에 있었다. 저장소 루트 바로 아래에는 `src` 폴더가 없으므로, setuptools의 패키지 자동탐색이 **아무 것도 찾지 못했다.**

결과적으로 editable 설치가 "성공"으로 끝났지만 실제로는 빈 패키지였다. 이는 다음 두 가지로 확인할 수 있다.

- `pip show <패키지명>` → `Editable project location`만 있고 실제 매핑이 없음
- `.venv/Lib/site-packages/ai_data_analysis-0.1.0.dist-info/top_level.txt`가 **빈 파일**
- `.venv/Lib/site-packages/__editable___ai_data_analysis_0_1_0_finder.py` 안의 `MAPPING: dict[str, str] = {}` — import를 가로챌 매핑 자체가 없음

## 3. 시도 2 — `pyproject.toml`을 실제 `src`가 있는 폴더로 이동

`where`가 실제 `src` 위치와 어긋나 있다는 게 원인이므로, `pyproject.toml` 자체를 `src`가 있는 `00_llm-data-analysis-course/` 폴더로 옮겼다.

In [ ]:
# 00_llm-data-analysis-course/pyproject.toml (이름만 변경, where는 동일하게 ".")

[project]
name = "00_llm-data-analysis-course"
...
[tool.setuptools.packages.find]
where = ["."]
include = ["course_utils*", "src*", "src."]

**결과**: 파일만 옮기고 다시 설치하지 않았더니 **여전히 같은 에러**가 발생했다.

**원인 해설**

`pip install -e .`는 실행하는 "그 순간"의 `pyproject.toml` 위치를 기준으로 site-packages(`.venv/Lib/site-packages`)에 매핑 정보를 새로 써준다. 파일을 옮기거나 내용을 수정하는 것만으로는 **이미 설치되어 있는 정보가 자동으로 갱신되지 않는다.** 즉 site-packages에는 여전히 "시도 1"에서 만들어진 빈 매핑이 그대로 남아 있었다.

➡️ **교훈**: `pyproject.toml`을 옮기거나 수정했다면, 반드시 **그 폴더 안에서** `pip install -e .`를 다시 실행해야 한다.

## 4. 재설치 후 검증

`00_llm-data-analysis-course` 폴더 **안으로 들어가서** 다시 설치했다.

In [ ]:
cd 00_llm-data-analysis-course
pip install -e .

**검증 방법**: `.venv/Lib/site-packages`의 editable finder 파일을 열어서 `MAPPING` 딕셔너리를 확인했다.

```python
MAPPING: dict[str, str] = {
    'course_utils': 'C:\\dev\\ai-data-analysis\\00_llm-data-analysis-course\\course_utils',
    'src': 'C:\\dev\\ai-data-analysis\\00_llm-data-analysis-course\\src',
}
```

→ 이번엔 제대로 채워졌다. 즉 **설치 위치(`pyproject.toml`이 있는 폴더 안에서 실행)만 맞으면 정상 동작**한다는 것을 확인했다.

**추가로 반드시 필요한 것 — 커널 재시작**

`.pth` 기반 editable 설치 훅은 파이썬 프로세스가 **시작될 때 한 번만** 등록된다(`site` 모듈이 처리). 이미 떠 있는 Jupyter 커널은 재설치 이전에 시작된 프로세스이므로, `pip install -e .`를 다시 실행해도 그 커널에는 반영되지 않는다. **반드시 커널을 Restart 해야 한다.**

## 5. 부작용 — 저장소 루트에 자꾸 생기는 `*.egg-info` 폴더

패키지 설치 방식을 쓰는 동안, 저장소 최상단에 `00_llm_data_analysis_course.egg-info/` 같은 폴더가 계속 생기는 문제가 있었다.

**원인 해설**

`pip install -e <경로>` 실행 시 setuptools가 빌드 과정에서 `<프로젝트명>.egg-info` 메타데이터 폴더를 만드는데, 이 폴더는 **`pyproject.toml` 위치 기준이 아니라, 명령을 실행한 시점의 현재 작업 디렉터리(cwd) 기준**으로 생성된다.

저장소 루트에 있는 상태에서
```
pip install -e ./00_llm-data-analysis-course
```
처럼 하위 폴더를 경로로 지정해서 실행하면, 대상 프로젝트는 하위 폴더의 것이지만 `egg-info` 폴더는 명령을 실행한 위치(루트)에 생긴다.

➡️ **교훈**: `pip install -e .`는 항상 **대상 폴더 안으로 `cd`한 뒤, 상대경로 `.`로** 실행해야 곁가지 폴더가 엉뚱한 곳에 생기지 않는다.

## 6. 최종 결정 — 패키지 설치 방식을 포기하고 `sys.path`에 직접 경로를 추가

`pyproject.toml`/editable install 방식은 다음과 같은 이유로 이 프로젝트(배포용 라이브러리가 아닌 학습용 분석 노트북 모음)에는 과했다.

- 파일 위치, 설치 실행 위치, 커널 재시작 여부에 따라 계속 어긋나기 쉬움
- 실수로 파일을 옮기면 아무 표시 없이 조용히 깨짐 (에러 메시지가 원인을 알려주지 않음)
- `egg-info` 같은 부산물이 엉뚱한 위치에 반복 생성됨

그래서 노트북 안에서 직접 `sys.path`에 프로젝트 루트를 추가하는, 패키징에 의존하지 않는 방식으로 바꿨다.

In [ ]:
import sys
import os

# 현재 노트북의 상위 폴더(=src, course_utils가 있는 프로젝트 루트)를 파이썬 검색 경로에 추가
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.preprocessing import (
    compare_shapes,
    preprocess_sales_data,
    validate_relationships,
)

**코드 해설**

- `os.getcwd()`: 노트북이 실행되는 현재 작업 디렉터리 (보통 노트북 파일이 있는 폴더, 예: `.../notebooks`)
- `os.path.join(os.getcwd(), '..')`: 그 상위 폴더 경로 문자열을 만든다 (`.../notebooks/..`)
- `os.path.abspath(...)`: 상대경로(`..`)가 섞인 문자열을 절대경로로 정리한다 (`.../00_llm-data-analysis-course`)
- `sys.path.append(...)`: 파이썬이 `import`할 때 뒤지는 경로 목록에 그 절대경로를 추가한다 → 이제 `src`, `course_utils`가 이 경로 바로 아래에 있으므로 import가 된다

**장점**: `pyproject.toml`, 설치 위치, 커널 재시작 여부와 무관하게 **노트북 셀을 실행하는 순간 항상 동작**한다. 패키지 설치 상태를 신경 쓸 필요가 아예 없어진다.

**주의사항 — 노트북 폴더 깊이가 다르면 `..` 개수를 맞춰야 한다**

이 코드는 노트북이 `00_llm-data-analysis-course/notebooks/` 바로 아래에 있다는 것을 전제로, 딱 한 단계(`..`)만 올라간다. 만약 노트북이 `notebooks/ch04/`처럼 한 단계 더 깊이 있다면 `../..`로 바꿔야 하고, 그렇지 않으면 프로젝트 루트가 아니라 `notebooks/`까지만 경로에 잡혀서 다시 `ModuleNotFoundError`가 난다.

폴더 깊이가 제각각인 노트북이 많다면, 아래처럼 `course_utils`(또는 `src`) 폴더가 나올 때까지 상위 폴더를 순회하며 찾는 방식이 더 안전하다 (프로젝트의 다른 노트북 `ch04_pandas_basic.ipynb`, `ch04/01.ipynb`에서 이미 쓰고 있던 방식).

In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()

# 현재 위치부터 상위 폴더들을 차례로 올라가며 course_utils 폴더를 찾는다
for path in (current, *current.parents):
    if (path / "course_utils").is_dir():
        project_root = path
        break
else:
    raise FileNotFoundError("course_utils 폴더를 찾지 못했습니다.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

## 7. 정리(clean-up) 작업

더 이상 쓰지 않기로 한 패키지 설치 관련 흔적을 모두 제거했다.

1. `pip uninstall ai-data-analysis "00_llm-data-analysis-course"` — `.venv`에 남아있던 editable 설치 2개 제거 (`__editable__*.pth`, finder 파일, `*.dist-info` 자동 삭제됨)
2. 루트의 `ai_data_analysis.egg-info/`, `00_llm_data_analysis_course.egg-info/` 폴더 삭제
3. 루트/하위 폴더의 `pyproject.toml` 삭제
4. `.gitignore`에 `*.egg-info/` 규칙 추가 — 앞으로 실수로 잘못된 위치에서 `pip install -e`를 실행해도 git에 걸리지 않도록 방지

이후 확인: `requirements.txt`에는 editable 설치(`-e .`) 줄이 없었으므로, 이 정리가 다른 설치 과정에 영향을 주지 않았다.

## 8. 다음에 같은 유형의 에러가 나면

| 증상 | 원인 체크리스트 | 해결 |
|---|---|---|
| `ModuleNotFoundError: No module named 'src'` (또는 `course_utils`) | 노트북에 `sys.path` 부트스트랩 코드가 있는가? 폴더 깊이에 맞게 `..` 개수(또는 parents 순회)가 맞는가? | 이 문서의 6번 코드를 노트북 맨 위 셀에 추가. 폴더 깊이가 다르면 `parents` 순회 버전 사용 |
| `pip install -e .` 후에도 import가 안 됨 | 어느 폴더 "안에서" 설치를 실행했는가? `pyproject.toml`을 옮긴 뒤 재설치를 했는가? | 대상 `pyproject.toml`이 있는 폴더로 `cd`한 뒤 `pip install -e .` 재실행 + 커널 Restart |
| 저장소 루트에 `*.egg-info` 폴더가 자꾸 생김 | `pip install -e ./어떤경로`처럼, 다른 위치에서 하위 폴더를 지정해 설치하고 있지 않은가? | 반드시 대상 폴더 "안"에서 `.`으로 설치. 지금은 패키지 설치 방식 자체를 쓰지 않기로 했으므로 발생하지 않아야 함 |

이 프로젝트는 현재 **패키지 설치(pyproject.toml/pip install -e) 방식을 쓰지 않고, 노트북마다 `sys.path` 부트스트랩 코드를 쓰는 것으로 최종 결정**되었다.

---

## 부록 — 같은 세션에서 함께 발견된 별개의 에러: `NameError: name 'order_status_sales' is not defined`

이 에러는 위 import 문제와는 **무관한, 별개의 원인**이었다.

In [ ]:
print(order_status_sales.columns)
print(products.head())

**발생한 에러**
```
NameError: name 'order_status_sales' is not defined
```

**원인 해설**

`order_status_sales`는 바로 앞의 다른 셀(`groupby`로 생성하는 셀)에서 정의된다. 그런데 에러의 트레이스백을 보면 이 셀이 **현재 커널 세션에서 다섯 번째로 실행된 셀**이었던 반면, `order_status_sales`를 만드는 셀에 저장되어 있던 출력은 **훨씬 이전(과거) 커널 세션에서 실행된 결과**였다.

즉, 커널을 재시작한 뒤 셀을 위에서부터 순서대로 실행하지 않고, `order_status_sales`를 만드는 셀을 건너뛴 채 이 셀만 따로 실행해서 생긴 문제였다. **노트북 로직 자체의 오류가 아니라 "커널 상태 ≠ 실행한 셀 순서" 불일치 문제.**

**해결**

Jupyter에서 **"Restart Kernel and Run All"**로 모든 셀을 위에서 아래로 순서대로 재실행. 낱개 셀만 실행할 때는 그 셀이 의존하는 앞 셀들을 먼저 실행했는지 항상 확인한다.

---

## 9. 심화 Q&A — `where`를 맞췄는데도 왜 계속 실패했나 / 프로젝트 루트가 여러 개면 이 방식은 절대 못 쓰는가

### Q1. `where` 경로를 맞췄는데도 실패가 반복된 이유는?

`pyproject.toml` 방식이 최종적으로 동작하려면 아래 **네 가지 조건이 동시에** 맞아야 한다.

1. `pyproject.toml`의 `where` 설정이 실제 `src` 위치와 일치
2. `pip install -e .`를 **그 파일이 있는 폴더 안에서** 실행
3. 파일을 옮기거나 수정했다면 **반드시 다시 설치** (재설치 없이는 이전 상태가 site-packages에 그대로 남음)
4. 이미 떠 있는 Jupyter 커널은 `.pth` 기반 import 훅을 다시 읽지 않으므로, 재설치 후 **커널 Restart**

이 네 개는 서로 독립적인 조건이라 하나라도 어긋나면 실패한다. 심지어 **1~3번을 완벽히 맞췄어도 4번(커널 재시작)을 하지 않으면 똑같은 에러가 재현**된다.

이 저장소 작업 과정에서 "where를 고쳤는데 또 실패했다"는 경험은, `where` 자체의 문제가 아니라 **이 네 가지 조건 중 하나가 계속 어긋나고 있었기 때문**이었다 (실제로 이후 파일을 다시 루트로 옮기면서 1번 조건이 또 깨졌다).

### Q2. 프로젝트 루트가 하나가 아닌 구조에서는 `pyproject.toml` 방식이 절대 안 되는가?

**아니다, 절대 불가능한 건 아니다.** 다만 현실적인 장애물이 두 가지 있다.

**장애물 A — 조건이 많아서 사람이 실수하기 쉬움**

바로 위 Q1의 네 가지 조건. "불가능"이 아니라 "귀찮고 잘 깨짐"의 문제다. 챕터 폴더 구조를 절대 안 바꾸고, 옮길 때마다 습관적으로 재설치+커널 재시작만 지키면 이론적으로는 계속 동작한다.

**장애물 B — 더 근본적인 위험, 이름 충돌**

이 저장소처럼 챕터 폴더 이름이 `src`, `course_utils`같이 **아주 일반적인 이름**인 경우, 다른 챕터(`01_python-basics`, `02_LLM_Date` 등)에도 각자 `src`라는 폴더가 있고 거기서도 `pyproject.toml`로 editable 설치를 한다면, 두 챕터의 패키지가 **같은 이름 `src`를 놓고 서로 자리를 뺏는** 문제가 생긴다. `.venv`가 저장소 전체에 딱 하나뿐이라서, 나중에 설치한 쪽이 먼저 설치된 쪽의 `src` 매핑을 덮어쓴다. 이건 위치를 아무리 정확히 맞춰도 피할 수 없는, **공유 가상환경 + 같은 이름 폴더**라는 구조 자체의 문제다.

**그래도 pyproject.toml 방식을 쓰고 싶다면 가능한 대안**

- 각 챕터 폴더 안 `src`를 `ch00_src`, `ch01_src`처럼 **챕터별로 고유한 이름**으로 바꾸고, 챕터마다 자기 `pyproject.toml`로 각각 editable 설치 → 이름 충돌 없이 여러 챕터가 공존 가능
- 챕터별로 **가상환경 자체를 분리**(챕터마다 `.venv`) → 서로 완전히 독립되어 충돌 자체가 안 생김
- `uv`, `Poetry`의 monorepo/workspace 기능처럼, 애초에 "하나의 레포 안 여러 프로젝트"를 지원하도록 설계된 도구를 사용

**결론**: 기술적으로 불가능한 게 아니라, "학습용으로 여러 챕터를 한 레포·한 가상환경에 몰아넣은 지금 구조"에서 이걸 안정적으로 굴리려면 들여야 하는 관리 비용(이름 규칙, 설치 규율)이 얻는 이득보다 커서 — 그래서 `sys.path` 방식으로 최종 결정한 게 이 상황에서는 합리적인 선택이었다.